# PDD Mobility Scanner — Slope Analysis

Step-by-step slope computation from `trail_data.csv` using gravity vector and GPS data.

**Upload your `trail_data.csv` when prompted.**

## Step 1: Load the CSV

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print(f"Loaded {len(df)} rows, {df['wp'].nunique()} waypoints")
print(f"Columns: {list(df.columns)}")
df.head()

## Step 2: Instantaneous slope from gravity vector

The BNO055 gravity vector `(gvx, gvy, gvz)` points toward Earth at ~9.81 m/s².

**Sensor mounting:** x-axis points down (`gvx ≈ -9.81` on flat ground), so:
- **Slope** = tilt of x-axis from vertical
- **Pitch** (forward/back) = from `gvz` component
- **Roll** (side tilt) = from `gvy` component

In [ ]:
# Gravity vector magnitude (should be ~9.81 for valid readings)
df['gv_mag'] = np.sqrt(df['gvx']**2 + df['gvy']**2 + df['gvz']**2)

# Total slope angle: angle between gravity vector and the x-axis (vertical in this mounting)
# acos(|gvx| / magnitude) gives tilt from vertical
df['slope_deg'] = np.degrees(np.arccos(np.clip(np.abs(df['gvx']) / df['gv_mag'], -1, 1)))

# Pitch (forward/back tilt) = gvz component relative to vertical axis gvx
# Roll (side tilt) = gvy component relative to vertical axis gvx
df['pitch_deg'] = np.degrees(np.arctan2(df['gvz'], np.abs(df['gvx'])))
df['roll_deg'] = np.degrees(np.arctan2(df['gvy'], np.abs(df['gvx'])))

print("Average gravity vector (sanity check):")
print(f"  gvx: {df['gvx'].mean():.2f}  (vertical axis)")
print(f"  gvy: {df['gvy'].mean():.2f}  (side)")
print(f"  gvz: {df['gvz'].mean():.2f}  (forward/back)")
print()
print("Sample-level slope stats:")
print(f"  Mean slope:  {df['slope_deg'].mean():.2f}°")
print(f"  Max slope:   {df['slope_deg'].max():.2f}°")
print(f"  Std dev:     {df['slope_deg'].std():.2f}°")
print()
print(df[['ms', 'gvx', 'gvy', 'gvz', 'slope_deg', 'pitch_deg', 'roll_deg']].head(10))

## Step 3: Average slope per waypoint segment

Each waypoint is a 5-second window with ~125 samples at 25Hz.
Averaging smooths out vibration and noise.

In [ ]:
# Aggregate per waypoint
wp_stats = df.groupby('wp').agg(
    slope_mean=('slope_deg', 'mean'),
    slope_max=('slope_deg', 'max'),
    slope_std=('slope_deg', 'std'),
    pitch_mean=('pitch_deg', 'mean'),
    roll_mean=('roll_deg', 'mean'),
    lat=('lat', 'last'),
    lng=('lng', 'last'),
    alt=('alt', 'last'),
    samples=('ms', 'count')
).reset_index()

print(f"{len(wp_stats)} waypoint segments:")
wp_stats

## Step 4: GPS-based grade between waypoints

Compare altitude change vs horizontal distance between consecutive GPS fixes.
This is noisier than the gravity vector but gives ground-truth grade over longer distances.

In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine_m(lat1, lon1, lat2, lon2):
    """Distance in meters between two GPS points."""
    R = 6371000
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

# Only use waypoints that have GPS data
gps_wp = wp_stats.dropna(subset=['lat', 'lng', 'alt']).copy()

# Compute distance and grade between consecutive waypoints
gps_wp['dist_m'] = 0.0
gps_wp['alt_change_m'] = 0.0
gps_wp['gps_grade_pct'] = 0.0
gps_wp['gps_slope_deg'] = 0.0

for i in range(1, len(gps_wp)):
    idx = gps_wp.index[i]
    prev = gps_wp.index[i-1]
    d = haversine_m(gps_wp.loc[prev, 'lat'], gps_wp.loc[prev, 'lng'],
                    gps_wp.loc[idx, 'lat'], gps_wp.loc[idx, 'lng'])
    alt_diff = gps_wp.loc[idx, 'alt'] - gps_wp.loc[prev, 'alt']
    gps_wp.loc[idx, 'dist_m'] = d
    gps_wp.loc[idx, 'alt_change_m'] = alt_diff
    if d > 0.5:  # only compute grade if we moved at least 0.5m
        gps_wp.loc[idx, 'gps_grade_pct'] = (alt_diff / d) * 100
        gps_wp.loc[idx, 'gps_slope_deg'] = np.degrees(np.arctan2(alt_diff, d))

print("GPS-based grade between waypoints:")
gps_wp[['wp', 'dist_m', 'alt_change_m', 'gps_grade_pct', 'gps_slope_deg', 'slope_mean']]

## Step 5: Visualize slope along the trail

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Raw 25Hz slope
axes[0].plot(df['ms'] / 1000, df['slope_deg'], alpha=0.3, linewidth=0.5, label='Raw 25Hz')
axes[0].set_ylabel('Slope (°)')
axes[0].set_title('Instantaneous slope from gravity vector')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Pitch and roll per waypoint
axes[1].bar(wp_stats['wp'] * 5, wp_stats['pitch_mean'], width=4, alpha=0.7, label='Pitch (fwd/back)')
axes[1].bar(wp_stats['wp'] * 5, wp_stats['roll_mean'], width=4, alpha=0.7, label='Roll (side)')
axes[1].set_ylabel('Angle (°)')
axes[1].set_title('Average pitch & roll per 5s segment')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# GPS altitude profile
if len(gps_wp) > 0:
    axes[2].plot(gps_wp['wp'] * 5, gps_wp['alt'], 'o-', label='GPS altitude')
    axes[2].set_ylabel('Altitude (m)')
    axes[2].set_title('GPS altitude profile')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

axes[2].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

## Step 6: Export results

In [ ]:
# Save waypoint-level slope summary
output = gps_wp[['wp', 'lat', 'lng', 'alt', 'dist_m', 'alt_change_m',
                  'slope_mean', 'slope_max', 'slope_std',
                  'pitch_mean', 'roll_mean',
                  'gps_grade_pct', 'gps_slope_deg', 'samples']].copy()

output.to_csv('slope_analysis.csv', index=False)
files.download('slope_analysis.csv')
print(f"Exported {len(output)} waypoint segments to slope_analysis.csv")